# Prompt Management with Langfuse

This notebook demonstrates **prompt management** using Langfuse for a stock price assistant chatbot.

## What you will learn

| Concept | Description |
|---------|-------------|
| **Template** | Prompts in Langfuse are templates with variables (e.g. `{{user_tickers}}`) that get filled at runtime via `.compile()` |
| **Messages List** | A chat prompt is a structured **list** of message dicts, not a flat string |
| **Role** | Each message carries a **role** (`system`, `user`, `assistant`) that shapes how the LLM interprets it |

## Project

A chatbot that checks stock prices using Yahoo Finance, personalized with the user's preferred ticker list.

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. You need Langfuse credentials in your `.env` file:
   - `LANGFUSE_SECRET_KEY`
   - `LANGFUSE_PUBLIC_KEY`
   - `LANGFUSE_HOST` (e.g. `https://cloud.langfuse.com`)
4. If packages are missing, run in your terminal:
   ```bash
   uv add langfuse yfinance
   ```

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langfuse import Langfuse
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

assert LANGFUSE_SECRET_KEY, "LANGFUSE_SECRET_KEY not found — add it to your .env file"
assert LANGFUSE_PUBLIC_KEY, "LANGFUSE_PUBLIC_KEY not found — add it to your .env file"

# ── Initialize Langfuse client ──────────────────────────────────────────────────
langfuse = Langfuse(
    secret_key=LANGFUSE_SECRET_KEY,
    public_key=LANGFUSE_PUBLIC_KEY,
    host=LANGFUSE_HOST,
)

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


## Stock Price Tool

Define a tool that fetches current stock prices using Yahoo Finance (`yfinance`).
The agent will call this tool when the user asks about stock prices.

In [2]:
@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, GOOGL, TSLA)
    Returns:
        Current stock price information including daily change
    """
    logger.info("get_stock_price called | ticker=%s", ticker)
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1d")
    if hist.empty:
        return f"Could not fetch price for {ticker}"
    price = hist["Close"].iloc[-1]
    prev_close = stock.info.get("previousClose")
    result = f"{ticker}: ${price:.2f}"
    if prev_close:
        change = price - prev_close
        pct = (change / prev_close) * 100
        result += f" (change: {change:+.2f}, {pct:+.2f}%)"
    logger.info("get_stock_price result: %s", result)
    return result


logger.info("Stock price tool ready")

INFO | Stock price tool ready


---

## 1. Template — Pull Prompt from Langfuse

A **prompt template** is a reusable prompt with **variables** (placeholders like `{{user_tickers}}`)
that get filled with real values at runtime.

In Langfuse, you create and version prompts in the UI, then fetch them in code:

```python
langfuse = Langfuse()
prompt = langfuse.get_prompt("my-prompt-name", type="chat")
```

The returned object is a `ChatPromptClient` — Langfuse's prompt wrapper that supports
`.compile()` to fill in variables.

Key differences from LangSmith:
- Variables use **double curly braces**: `{{user_tickers}}` (not `{user_tickers}`)
- Prompts are fetched with `get_prompt()` (not `pull_prompt()`)
- The `type="chat"` parameter tells Langfuse to return a chat prompt (list of messages)

> **Prerequisite:** Create the prompt in the Langfuse UI before running the next cell.
> Update `PROMPT_NAME` below to match your prompt's name.

In [ ]:
# ── Pull prompt from Langfuse ─────────────────────────────────────────────────
PROMPT_NAME = "stock-price-assistant"  # ← replace with your prompt name

prompt = None # TODO: load prompt from langfuse

# The prompt is a TEMPLATE — it has placeholders that we fill at runtime
print(f"Prompt name : {prompt.name}")
print(f"Version     : {prompt.version}")
print(f"Type        : {type(prompt).__name__}")
print(f"Raw prompt  : {prompt.prompt}")

Prompt name : stock-price-assistant
Version     : 1
Type        : ChatPromptClient
Raw prompt  : [{'type': 'message', 'role': 'system', 'content': "Answer user's question"}]


---

## 2. Messages List — A Prompt is a List of Messages

Unlike a simple string prompt, a **chat prompt** is a structured **list of messages**.
Each entry is a dict with `role` and `content` keys.

```python
prompt.prompt  # → [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
```

This matters because:
- Each message can have its own template variables
- The **order** of messages matters — the LLM reads them top to bottom
- You can mix system instructions, example conversations, and user input in one prompt

In [ ]:
# ── Inspect the messages list ──────────────────────────────────────────────────
messages = prompt.prompt

print(f"Number of messages in prompt: {len(messages)}\n")

for i, msg in enumerate(messages):
    role = msg["role"]
    content = msg["content"]
    preview = content[:120] + ("..." if len(content) > 120 else "")
    print(f"  [{i}] Role: {role}")
    print(f"      Content: \"{preview}\"")
    print()
# TODO: see how menay messages and role in the prompt, compare with the UI

Number of messages in prompt: 1

  [0] Role: system
      Content: "Answer user's question"



---

## 3. Roles — Each Message Has a Role

Every message in a chat prompt has a **role** that tells the LLM how to interpret it:

| Role | Purpose | Example |
|------|---------|--------|
| **system** | Sets the agent's behavior, personality, and constraints | *"You are a stock assistant…"* |
| **user** | Represents user input | *"What's the price of AAPL?"* |
| **assistant** | Represents assistant responses (useful for few-shot examples) | *"AAPL is currently at $150.00"* |

The **system** role is especially powerful — it shapes the entire conversation.
This is where we inject the user's preferred tickers via the `{{user_tickers}}` template variable.

> **Note:** Langfuse uses standard OpenAI role names (`system`, `user`, `assistant`)
> while LangChain uses (`system`, `human`, `ai`).

In [5]:
# ── Show the role of each message ──────────────────────────────────────────────
print("Message roles in this prompt:\n")
for i, msg in enumerate(prompt.prompt):
    role = msg["role"]
    content_preview = msg["content"][:80] + ("..." if len(msg["content"]) > 80 else "")
    print(f"  [{i}] Role: {role:<12} | Content: \"{content_preview}\"")

Message roles in this prompt:

  [0] Role: system       | Content: "Answer user's question"


---

## Format the Template

Now we **compile** the template by filling in the variables with real values.
The user's preferred tickers are injected into the `{{user_tickers}}` variable.

Langfuse's `.compile()` method replaces `{{variable}}` placeholders and returns
a list of message dicts — ready to send to the LLM.

```python
compiled = prompt.compile(user_tickers="AAPL, GOOGL, TSLA")
# → [{"role": "system", "content": "You are a stock assistant. User's tickers: AAPL, GOOGL, TSLA"}, ...]
```

In [ ]:
# ── User's preferred tickers ───────────────────────────────────────────────────
USER_TICKERS = ["AAPL", "GOOGL", "TSLA"]

compiled_messages = None # TODO: compile the prompt with the user's tickers to create a compiled list of messages

print(f"Type: {type(compiled_messages).__name__}")
print(f"Number of compiled messages: {len(compiled_messages)}\n")

for i, msg in enumerate(compiled_messages):
    print(f"  [{i}] Role: {msg['role']}")
    print(f"      Content: {msg['content']}")
    print()

Type: list
Number of compiled messages: 1

  [0] Role: system
      Content: Answer user's question



---

## Build the Agent

Wire everything together:
1. The **compiled system prompt** (with tickers injected) tells the agent who it is
2. The **stock price tool** gives the agent the ability to look up real prices
3. `create_agent` builds a ReAct loop that reasons and acts autonomously

In [ ]:
# ── Extract the system message content from compiled messages ────────────────────
system_content = next(
    msg["content"] for msg in compiled_messages if msg["role"] == "system"
)

# ── Build the agent with the formatted prompt ─────────────────────────────────
agent = create_agent(
    model=llm,
    tools=[get_stock_price],
    system_prompt=system_content, # TODO: remove system prompt here and add compiled messages later
)

logger.info("Agent created — prompt injected, tools: %s", [get_stock_price.name])

INFO | Agent created — prompt injected, tools: ['get_stock_price']


---

## Test the Agent

Ask the agent about the user's preferred stocks.
Because the system prompt contains the ticker list, the agent knows which stocks to check.

In [8]:
# ── Test 1: Ask about preferred stocks ─────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="How are my preferred stocks doing today?")]}
)

print(result["messages"][-1].content)

Please provide me with the ticker symbols of your preferred stocks, and I'll check their current stock prices for you.


In [10]:
# ── Test 2: Ask about a specific stock ─────────────────────────────────────────
result2 = agent.invoke(
    {"messages": [HumanMessage(content="What is the current price of Microsoft?")]}
)

print(result2["messages"][-1].content)

INFO | get_stock_price called | ticker=MSFT
INFO | get_stock_price result: MSFT: $356.77 (change: -9.20, -2.51%)


The current price of Microsoft (MSFT) is $356.77, with a change of -$9.20, which is a decrease of 2.51%.


---

## Message Placeholders — Insert Chat History into a Prompt

A **message placeholder** lets you inject a **list of messages** (e.g. conversation history)
at a specific position in a chat prompt template.

Instead of a regular `{role, content}` message, you define a placeholder entry:

```json
{ "type": "placeholder", "name": "chat_history" }
```

At runtime, `.compile()` accepts **both** template variables and placeholder values:

```python
prompt.compile(
    user_tickers="AAPL, GOOGL",        # template variable
    chat_history=[                       # placeholder — list of messages
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "..."},
    ],
)
```

> **Prerequisite:** Add a message placeholder named `chat_history` to your
> `stock-price-assistant` prompt in the Langfuse UI (use the *Add message placeholder* button).

In [ ]:
# ── Re-fetch the prompt (now with the chat_history placeholder) ───────────────
prompt_with_ph = langfuse.get_prompt(PROMPT_NAME, type="chat", cache_ttl_seconds=0)

logger.info("Raw prompt messages:")
for i, msg in enumerate(prompt_with_ph.prompt):
    logger.info("  [%d] %s", i, msg)

In [ ]:
# ── Compile with a short chat history about Microsoft ─────────────────────────
chat_history = [
    {"role": "user", "content": "I've been looking at Microsoft lately. Seems like a solid company."},
    {"role": "assistant", "content": "Microsoft is indeed one of the top tech companies, with strong cloud and AI businesses."},
    {"role": "user", "content": "What is the current price of Microsoft?"},
]

compiled_with_history = None # TODO: compile the prompt with the chat history to create a compiled list of messages

logger.info("Compiled messages with chat history:")
for i, msg in enumerate(compiled_with_history):
    preview = msg["content"][:100] + ("..." if len(msg["content"]) > 100 else "")
    logger.info("  [%d] %-9s | %s", i, msg["role"], preview)

In [ ]:
# ── Run the agent with the compiled messages (history + final question) ────────
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

langchain_messages = []
for msg in compiled_with_history:
    if msg["role"] == "system":
        langchain_messages.append(SystemMessage(content=msg["content"]))
    elif msg["role"] == "assistant":
        langchain_messages.append(AIMessage(content=msg["content"]))
    else:
        langchain_messages.append(HumanMessage(content=msg["content"]))

result_ph = agent.invoke({"messages": langchain_messages})
print(result_ph["messages"][-1].content)

---

## Using Model Config from a Langfuse Prompt

When you create a prompt in the Langfuse UI, you can attach a **JSON config** object
alongside the template (e.g. `{"temperature": 0.7, "model": "gpt-4o-mini", "max_tokens": 1024}`).

At runtime, this config is available via `prompt.config`:

```python
prompt = langfuse.get_prompt("my-prompt", type="chat")
prompt.config  # → {"temperature": 0.7, "model": "gpt-4o-mini"}
```

This is the Langfuse equivalent of LangSmith's `include_model=True` — it lets
**non-developers** (e.g. prompt engineers) control model parameters directly from the UI
without any code changes.

> **Prerequisite:** Add a JSON config to your `stock-price-assistant` prompt in the
> Langfuse UI (click the **Config** tab when editing the prompt).

In [ ]:
# ── Pull prompt and inspect its config ─────────────────────────────────────────
prompt_with_config = langfuse.get_prompt(PROMPT_NAME, type="chat", cache_ttl_seconds=0)

config = prompt_with_config.config
print(f"Prompt config: {config}\n")

if not config:
    print("⚠ No config found — add a JSON config in the Langfuse UI for this prompt.")
else:
    for key, value in config.items():
        print(f"  {key:<15}: {value}")

# ── Use config values to create an LLM ────────────────────────────────────────
pulled_temperature = None # TODO: pull the temperature from the config
pulled_model = None # TODO: pull the model from the config
logger.info("Using temperature=%.2f from Langfuse prompt config", pulled_temperature)

if MODEL_PROVIDER == "openai":
    llm_from_config = ChatOpenAI(
        model=pulled_model or "gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=pulled_temperature,
    )
else:
    llm_from_config = ChatGoogleGenerativeAI(
        model=pulled_model or "gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=pulled_temperature,
    )

print(f"\nCreated LLM with config → model={llm_from_config.model_name}, temperature={pulled_temperature}")

# ── Build an agent using the config-driven LLM ───────────────────────────────
system_content = next(
    msg["content"]
    for msg in prompt_with_config.compile(user_tickers=", ".join(USER_TICKERS))
    if msg["role"] == "system"
)

agent_from_config = create_agent(
    model=llm_from_config,
    tools=[get_stock_price],
    system_prompt=system_content,
)

result_config = agent_from_config.invoke(
    {"messages": [HumanMessage(content="What is the current price of AAPL?")]}
)
print(f"\nAgent response:\n{result_config['messages'][-1].content}")

---

## Wrap-up

### What we covered

| Concept | What you saw |
|---------|-------------|
| **Template** | `prompt.prompt` — the prompt has `{{user_tickers}}` filled at runtime via `.compile()` |
| **Messages List** | `prompt.prompt` — a prompt is a list of message dicts, not a flat string |
| **Role** | Each message has a role (`system`, `user`, `assistant`) that shapes LLM behavior |
| **Placeholder** | `{ "type": "placeholder", "name": "chat_history" }` — inject a list of messages (e.g. conversation history) at a specific position in the prompt |

### Key takeaways

- **Langfuse** stores prompts centrally — update them in the UI without redeploying code
- **Templates** use `{{variable}}` syntax and `.compile()` to inject user-specific data at runtime
- **Placeholders** let you insert entire conversation histories into a prompt at a designated position
- **Roles** control how the LLM interprets each message — the `system` role is the most powerful lever
- **Version control** — Langfuse tracks prompt versions and supports labels (`production`, `staging`, `latest`)
- **Observability** — link prompts to traces to analyze performance by prompt version